In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import RandomizedSearchCV

import warnings

In [4]:
def evaluate_model(true, predicted, model, X_test):
    precision = precision_score(true, predicted , zero_division = 0)
    recall = recall_score(true, predicted , zero_division = 0)
    f1 = f1_score(true , predicted, zero_division = 0)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(true, y_pred_proba)
    pr_auc = average_precision_score(true, y_pred_proba)
    return precision, recall, f1, roc_auc, pr_auc

In [ ]:
def classification(X_train, y_train, X_test, y_test):

    # At 2% blunder rate: ~49 non-blunders per blunder
    y_train_blunder = y_train['is_blunder']
    y_test_blunder  = y_test['is_blunder']

    ratio = (y_train_blunder == 0).sum() / (y_train_blunder == 1).sum()

    models = {
        # class_weight='balanced' automatically computes inverse frequency weights
        "Logistic Regression": LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
        ),

        # Same as above — class_weight works identically for Lasso/Ridge variants
        "Lasso": LogisticRegression(
            penalty='l1', solver='liblinear',
            class_weight='balanced',
            max_iter=1000,
        ),

        "Ridge": LogisticRegression(
            penalty='l2', solver='liblinear',
            class_weight='balanced',
            max_iter=1000,
        ),

        # class_weight='balanced' applies inverse frequency to each split criterion
        "Decision Tree": DecisionTreeClassifier(
            class_weight='balanced',
            max_depth=6,          # prevent overfitting on minority class
        ),

        "Random Forest Classifier": RandomForestClassifier(
            class_weight='balanced',
            n_estimators=300,
            max_depth=8,
            random_state=42,
        ),

        # scale_pos_weight is XGBoost's equivalent of class_weight
        # = number of negative samples / number of positive samples
        "XGBClassifier": XGBClassifier(
            scale_pos_weight=ratio,
            eval_metric='aucpr',   # optimize for PR AUC
            n_estimators=300,
            random_state=42,
            verbosity=0,
        ),

        # CatBoost equivalent — scale_pos_weight works the same way
        "CatBoosting Classifier": CatBoostClassifier(
            scale_pos_weight=ratio,
            eval_metric='AUC',
            iterations=300,
            random_seed=42,
            verbose=False,
        ),

        # AdaBoost has no direct class_weight or scale_pos_weight
        # Use a weighted base estimator instead
        "AdaBoost Classifier": AdaBoostClassifier(
            estimator=DecisionTreeClassifier(
                class_weight='balanced',
                max_depth=2,
            ),
            n_estimators=200,
            random_state=42,
        ),
    }
 
    # Lists to store results
    results_train = []
    results_test = []
    
    for model_name, model in models.items():
 
        # Train model
        model.fit(X_train, y_train)
        
        # Training predictions
        y_train_pred = model.predict(X_train)
        accuracy_train = accuracy_score(y_train, y_train_pred)
        train_precision, train_recall, train_f1, train_roc_auc, train_pr_auc = evaluate_model(
            y_train, y_train_pred, model, X_train
        )
        
        # Test predictions
        y_test_pred = model.predict(X_test)
        accuracy_test = accuracy_score(y_test, y_test_pred)
        test_precision, test_recall, test_f1, test_roc_auc, test_pr_auc = evaluate_model(
            y_test, y_test_pred, model, X_test
        )
        
        # Store training results
        results_train.append({
            'Model': model_name,
            'Accuracy': accuracy_train,
            'Precision': train_precision,
            'Recall': train_recall,
            'F1 Score': train_f1,
            'ROC AUC': train_roc_auc,
            'PR AUC': train_pr_auc
        })
        
        # Store test results
        results_test.append({
            'Model': model_name,
            'Accuracy': accuracy_test,
            'Precision': test_precision,
            'Recall': test_recall,
            'F1 Score': test_f1,
            'ROC AUC': test_roc_auc,
            'PR AUC': test_pr_auc
        })
    
    # Create DataFrames
    df_train = pd.DataFrame(results_train)
    df_test = pd.DataFrame(results_test)
    
    # Display tables
    print("=" * 120)
    print("TRAINING SET PERFORMANCE")
    print("=" * 120)
    print(df_train.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    print("\n")
    
    print("=" * 120)
    print("TEST SET PERFORMANCE")
    print("=" * 120)
    print(df_test.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    print("\n")

In [ ]:
scaled_X_train=pd.read_csv('model_data/scaled_blunder_blitz_X_train.csv')
y_train=pd.read_csv('model_data/blunder_blitz_y_train.csv')

In [ ]:
scaled_X_test=pd.read_csv('model_data/scaled_blunder_blitz_X_test.csv')
y_test=pd.read_csv('model_data/blunder_blitz_y_test.csv')

In [8]:
classification(scaled_X_train, y_train, scaled_X_test, y_test)

c:\Users\steve\anaconda3\envs\chess\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\steve\anaconda3\envs\chess\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\steve\anaconda3\envs\chess\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\steve\anaconda3\envs\chess\Lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please chan

TRAINING SET PERFORMANCE
                   Model  Accuracy  Precision  Recall  F1 Score  ROC AUC  PR AUC
     Logistic Regression    0.8949     0.2221  0.9289    0.3585   0.9496  0.2520
                   Lasso    0.8948     0.2221  0.9289    0.3584   0.9496  0.2518
                   Ridge    0.8948     0.2218  0.9274    0.3580   0.9496  0.2521
           Decision Tree    0.8863     0.2176  1.0000    0.3574   0.9534  0.2697
Random Forest Classifier    0.9027     0.2453  1.0000    0.3939   0.9766  0.5541
           XGBClassifier    1.0000     1.0000  1.0000    1.0000   1.0000  1.0000
  CatBoosting Classifier    0.9684     0.5000  1.0000    0.6667   0.9987  0.9452
     AdaBoost Classifier    0.9677     0.4948  1.0000    0.6620   0.9987  0.9608


TEST SET PERFORMANCE
                   Model  Accuracy  Precision  Recall  F1 Score  ROC AUC  PR AUC
     Logistic Regression    0.8989     0.2287  0.9273    0.3669   0.9483  0.2436
                   Lasso    0.8991     0.2290  0.9273    0.36